In [ ]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Hybrid variable resolution
catalog = os.getenv("DATABRICKS_BUNDLE_VAR_catalog")
schema = os.getenv("DATABRICKS_BUNDLE_VAR_schema")

try:
    if not catalog:
        catalog = dbutils.widgets.get("catalog")
    if not schema:
        schema = dbutils.widgets.get("schema")
except Exception:
    pass

if not catalog:
    raise ValueError("❌ No catalog provided via env or job parameters.")

input_table = f"{catalog}.{schema}.etl_demo_output"
output_table = f"{catalog}.{schema}.sales_transformed"

print(f"➡️ Using catalog={catalog}, schema={schema}")
print(f"Reading input table: {input_table}")

df = spark.read.table(input_table)
df_filtered = df.filter(df["amount_with_tax"] > 150)
df_filtered.write.mode("overwrite").saveAsTable(output_table)

print(f"✅ Transformed sales table created at {output_table}")